# G1 · Validación de extracción

**Spec:** [`docs/spec_G1_codex_extraction_validation.md`](../docs/spec_G1_codex_extraction_validation.md)  |  **Bloque:** G · Caracterización  |  **Run de este set:** `ROXs12b_realigned`

Valida los métodos de extracción: covarianza espectral, inflación espacial, presupuesto de sesgo.

| | |
|---|---|
| **Entrada** | Controles + E4 grid |
| **Salida (QC/productos)** | `stages/stage_g1_qc.json`, `g1_channel_covariance.npz` |
| **Consume aguas abajo** | G2, E3 (throughput/bias) |


## Qué hace G1 y qué valida

G1 **valida los métodos de extracción**: cuantifica la **covarianza del ruido** (espectral + espacial), el **presupuesto de sesgo**, y emite un veredicto por método.

**Covarianza (por qué el ruido debe ser empírico):**
- **Espectral:** longitud de correlación 2.34 canales, **n_eff/n = 0.428** — solo esa fracción de los canales cuenta como independiente; promediar en λ NO gana √N.
- **Espacial:** sumar en una caja N×N infla la varianza frente a la suma ingenua √N: **box3 ≈ 5.4×, box5 ≈ 18.7×**. Es la **covarianza del remuestreo** — el **mismo** fenómeno detrás de M5 (STAT subestimado). G1 es donde se caracteriza del todo.

**Presupuesto de sesgo:** los dos métodos validados (psffit, optimal_psfsub) tienen ~**33% de pérdida de throughput** (que E3 corrige dividiendo por el throughput); el término de PSF es ~0 (la perturbación de PSF es un no-op, la PSF está flux-normalizada).

**Veredictos:** **psffit & optimal_psfsub = `validated_with_bias`** (pérdida de throughput medida y corregida aguas abajo); **aperture & optimal_ls = `rejected`** (insensibles en el borde del compañero). Es exactamente lo que D1 usa: el **par primario = los dos validados**.

**Nota de dominio:** la covarianza de los controles **NO** captura el sistemático de halo en la posición del compañero — eso es el presupuesto de inyección (el B6 de D1).


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python scripts/run_g1.py --run-id $RUN
```

Moderado.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_g1_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python scripts/run_g1.py --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_g1_qc.json', RUN_ID)
nb.show(qc, keys=['method_verdicts', 'corr_length_channels_median', 'n_eff_over_n_median', 'spatial_inflation_by_box'], title='G1')


## Resultados que llevaron a la conclusión

Veredictos, covarianza y presupuesto de sesgo del `stage_g1_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('G1', 'stages/stage_g1_qc.json'):
        import pandas as pd
        q = nb.load_qc('stages/stage_g1_qc.json', RUN_ID)
        cov = q['covariance']
        print('veredictos por método:')
        for m, v in q['method_verdicts'].items():
            print(f"   {m:15s} {v}")
        print(f"\ncovarianza espectral: corr_length {cov['corr_length_channels_median']:.2f} ch, "
              f"n_eff/n = {cov['n_eff_over_n_median']:.3f} (rho_1 {cov['rho_1_median']:.2f})")
        print('inflación espacial por caja:', {k: round(v, 1) for k, v in cov['spatial_inflation_by_box'].items()})
        print()
        d = pd.read_csv(nb.run_dir(RUN_ID) / 'tables' / 'g1_bias_budget.csv')
        tot = d[d.term == 'TOTAL']
        print('presupuesto de sesgo (TOTAL throughput loss):')
        for _, r in tot.iterrows():
            v = 'rechazado (insensible)' if pd.isna(r['value_frac']) else f"{r['value_frac']*100:+.0f}%"
            print(f"   {r['method']:15s} {v}")


## Plot 1 — la covarianza del remuestreo (por qué σ es empírico)

Inflación de la varianza al sumar en cajas N×N frente a la suma ingenua √N (=1). En este objeto: box3 ≈ 5.4×, box5 ≈ 18.7× → un σ de apertura de √(Σ STAT) está mal por estos factores. Es el mismo mecanismo que M5. (Espectral: n_eff/n = 0.428.)


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_g1_qc.json', RUN_ID); cov = q['covariance']
    infl = cov['spatial_inflation_by_box']
    boxes = sorted(infl, key=lambda k: int(k)); xs = [f'{int(b)}×{int(b)}' for b in boxes]; vals = [infl[b] for b in boxes]
    fig, ax = plt.subplots(figsize=(8.5, 4.3))
    ax.bar(xs, vals, color='tab:orange', alpha=0.85)
    for i, v in enumerate(vals): ax.text(i, v + 0.3, f'{v:.1f}×', ha='center', fontsize=9)
    ax.axhline(1.0, color='tab:green', ls='--', label='suma ingenua √N (=1)')
    ax.set_xlabel('caja de integración (N×N spaxels)'); ax.set_ylabel('inflación varianza real / ingenua')
    ax.set_title(f"G1 · covarianza del remuestreo (espectral n_eff/n={cov['n_eff_over_n_median']:.2f}, "
                 f"corr_length {cov['corr_length_channels_median']:.1f} ch)")
    ax.legend(fontsize=8); fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'g1_validation'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'covariance.png', dpi=110); print('figura ->', outdir / 'covariance.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — veredictos y pérdida de throughput por método

psffit y optimal_psfsub = `validated_with_bias` (pérdida de throughput ~−33%, corregida en E3); aperture y optimal_ls = `rejected` (insensibles en el borde, throughput no medible). Son los dos validados que D1 compara.


In [ ]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_g1_qc.json', RUN_ID); ver = q['method_verdicts']
    d = pd.read_csv(nb.run_dir(RUN_ID) / 'tables' / 'g1_bias_budget.csv')
    tot = d[d.term == 'TOTAL'].set_index('method')['value_frac']
    methods = ['psffit', 'optimal_psfsub', 'aperture', 'optimal_ls']
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    for i, m in enumerate(methods):
        val = tot.get(m, np.nan)
        if np.isnan(val):
            ax.text(i, 0.02, 'rechazado\n(insensible)', ha='center', va='bottom', fontsize=9, color='tab:red')
        else:
            ax.bar(i, abs(val) * 100, color='tab:green')
            ax.text(i, abs(val) * 100 + 1, f'{val*100:+.0f}%\n{ver[m]}', ha='center', fontsize=8)
    ax.set_xticks(range(len(methods))); ax.set_xticklabels(methods, fontsize=9)
    ax.set_ylabel('|pérdida de throughput| [%]'); ax.set_ylim(0, 45)
    ax.set_title('G1 · veredictos: psffit/psfsub validados (−33%, corregido en E3); aperture/ls rechazados')
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'g1_validation'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'verdicts.png', dpi=110); print('figura ->', outdir / 'verdicts.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- psffit & optimal_psfsub = `validated_with_bias` (throughput loss ~−33%, corregido en E3); aperture & optimal_ls = `rejected` (insensibles en el borde). Es el par primario de D1. · [`docs/noise_model.md`](../docs/noise_model.md)
- Correlación espectral 2.34 ch, **n_eff/n=0.428**; inflación espacial box3≈5.4×, box5≈18.7× → **confirma M5** y justifica el ruido empírico.
- La covarianza de controles NO captura el sistemático de halo en la posición del compañero (eso es el B6 / presupuesto de inyección).


## Conclusión (registrada)

**G1: valida psffit & optimal_psfsub (el par primario de D1); rechaza aperture & optimal_ls.**

- **Covarianza espectral:** corr_length 2.34 ch, n_eff/n 0.428 (promediar en λ no gana √N).
- **Covarianza espacial:** box3 5.4×, box5 18.7× → **confirma M5** (STAT subestimado), justifica el σ empírico.
- **Sesgo:** ~−33% pérdida de throughput en los validados (corregido en E3); término de PSF ~0.
- **Downstream:** los veredictos definen el par primario que D1 compara y los throughputs que E3 aplica.
